<a href="https://colab.research.google.com/github/leilagharavi/analytics-test-case/blob/main/walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Test Case for Marketing Analytics

# Information
- **Author:** Leila Gharavi
- **Email:** `gharavi95@gmail.com`
- **Date:** 18 Nov 2026

#### Relevant links
- [Google Merchandize Store](https://merch.google/$0)
- BigQuery dataset: `bigquery-public-data:ga4
obfuscated_sample_ecommerce`
- GCP project: `crystalloids-analytics`
  - Dataset `analytics` created with US location (due to raw source origin)
  - Dataset `ml` created with US location (due to raw source origin)
- Access to datasets: `BigQuery Data Viewer`
- [GitHub repo](https://github.com/leilagharavi/analytics-test-case.git)


## Timeline
- 10:00 - 10:30 Familirized with hardware setup
- 10:30 - 11:00 Setting up notebook skeleton, GCP project, and gitHub repository
- 11:00 - 11:30 Install dbt and set up dbt project, [see here](https://colab.research.google.com/github/leilagharavi/analytics-test-case/blob/main/walkthrough.ipynb#scrollTo=hsmD51nKIOni&line=6&uniqifier=1)
- 11:30 - 12:00 Switch to local dbt development and set up BQ connection with SA
- 12:00 - 13:00 Set up dbt project and data models, [see here](https://colab.research.google.com/github/leilagharavi/analytics-test-case/blob/main/walkthrough.ipynb#scrollTo=mR2Ds6hhPw--&line=26&uniqifier=1)
- 13:00 - 13:30 Pre-training analysis of the features, [see here](https://colab.research.google.com/github/leilagharavi/analytics-test-case/blob/main/walkthrough.ipynb#scrollTo=sZjzbSXGi3zB&line=3&uniqifier=1)
- 13:30 - 14:00 ML model training and evaluation, [see here](https://colab.research.google.com/github/leilagharavi/analytics-test-case/blob/main/walkthrough.ipynb#scrollTo=x24ptpt5pNpx&line=7&uniqifier=1)
- 14:00 - 14:30 Create prediction products, [see here](https://colab.research.google.com/github/leilagharavi/analytics-test-case/blob/main/walkthrough.ipynb#scrollTo=MtgLqIuzwPrX&line=35&uniqifier=1)
- 14:30 - 15:00 Wrap up and access control

## Process

### dbt

#### Installation





In [ ]:
# Uninstall any conflicting packages if present
!pip uninstall -y dbt-bigquery sqlglot dbt-state

# Install dbt-bigquery cleanly with user permissions
!pip install --user dbt-bigquery

## Check dbt version
!dbt --version

#### dbt project set up
- I choose to keep the notebook file in the same repo where the dbt project lives.

In [ ]:
!dbt init marketing_analytics

### Local dbt setup

- Service Account: `marketing_analytics_sa`
- Permissions: `BigQuery Admin` and `BigQuery Job User`
- JSON key downloaded on local (independant) path
```Bash
dbt init marketing_analytics
dbt debug
```
### Create source (raw layer)
```Bash
touch marketing_analytics/seeds/sources.yml
```
Which contains the following code:
```YAML
version: 2

sources:
  - name: ga4_raw
    database: bigquery-public-data
    schema: ga4_obfuscated_sample_ecommerce
    tables:
      - name: events
        identifier: events_*
```
### Create layers

#### Staging layer
- The data is cleaned and pre-processed before implementing business logic in the next layers.
- Materialized as view to limit computational usage.
```Bash
mkdir marketing_analytics/models/staging/
touch marketing_analytics/models/staging/stg_ga4_events.sql
```
The dbt tests for this model is created by running
```Bash
touch marketing_analytics/models/staging/schema.yml
```
which includes the following code:
```YAML
version: 2

models:
  - name: stg_ga4_events
    description: "Cleaned and flattened GA4 event-level staging model."
    columns:
      - name: user_pseudo_id
        description: "Unique identifier for the user."
        tests:
          - not_null
          
      - name: event_timestamp
        description: "Exact timestamp of the event."
        tests:
          - not_null

      - name: event_name
        description: "The name of the GA4 event logged."
        tests:
          - not_null
```

#### Marts layer
- Materialized as table and includes features for ML model training.
```Bash
mkdir marketing_analytics/models/marts/
touch marketing_analytics/models/marts/user_features.sql
touch marketing_analytics/models/marts/schema.yml
```
The model is created as
```SQL
SELECT
    user_pseudo_id,
    
    -- Behavioral Aggregations
    COUNT(DISTINCT session_id) AS total_sessions,
    COUNT(1) AS total_events,
    COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN session_id END) AS purchase_session_count,
    COALESCE(SUM(event_value), 0) AS total_spend_usd,
    
    -- Contextual Attributes (Taking the first/any value for user demographics)
    ANY_VALUE(device_category) AS device_category,
    ANY_VALUE(country) AS country,
    
    -- Machine Learning Target Label: Did the user make a purchase? (1 or 0)
    IF(COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN session_id END) > 0, 1, 0) AS label_made_purchase

FROM {{ ref('stg_ga4_events') }}
GROUP BY user_pseudo_id
```
Moreover, a schema file is created to include tests and descriptions as follows:

```YAML
version: 2

models:
  - name: user_features
    description: "User-level aggregated feature table built from GA4 staging events, optimized for BQML and user behavior analytics."
    columns:
      - name: user_pseudo_id
        description: "Primary key: Unique pseudo identifier for each user."
        tests:
          - unique
          - not_null
          
      - name: total_sessions
        description: "Total distinct sessions initiated by the user."
        tests:
          - not_null
          
      - name: label_made_purchase
        description: "Binary target label for BQML: 1 if the user purchased, 0 otherwise."
        tests:
          - not_null
          - accepted_values:
              values: [0, 1]
              quote: false
```
Note that the `quote: false` in the end was added to avoid errors due to string mishandling and type mismatch.





### User feature analysis

I ran the following query to inspect the user features in the marts layer before feeding in to the ML training. Currently, it lives in the `analyses` directory as `traffic_source_performance.sql`.
```SQL
WITH user_traffic AS (
    SELECT
        user_pseudo_id,
        COALESCE(ANY_VALUE(traffic_source), '(direct)') AS traffic_source,
        COALESCE(ANY_VALUE(traffic_medium), '(none)') AS traffic_medium
    FROM `crystalloids-analytics.analytics.stg_ga4_events`
    GROUP BY user_pseudo_id
),

user_metrics AS (
    SELECT
        user_pseudo_id,
        total_sessions,
        total_spend_usd,
        label_made_purchase
    FROM `crystalloids-analytics.analytics.user_features`
)

SELECT
    t.traffic_source,
    t.traffic_medium,
    COUNT(DISTINCT u.user_pseudo_id) AS total_users,
    SUM(u.label_made_purchase) AS purchasing_users,
    ROUND(SAFE_DIVIDE(SUM(u.label_made_purchase), COUNT(DISTINCT u.user_pseudo_id)) * 100, 2) AS conversion_rate_pct,
    ROUND(SUM(u.total_spend_usd), 2) AS total_revenue_usd

FROM user_traffic t
JOIN user_metrics u ON t.user_pseudo_id = u.user_pseudo_id
GROUP BY 1, 2
ORDER BY total_revenue_usd DESC
```
#### Takeaway
- Noticed with possibly unique campaign tags being `traffic_source`. This lead to adding the following logic to the satging model.
```SQL
cleaned AS (
    SELECT
        *,
        -- Cleaned Traffic Source (handling safeframe/googlesyndication noise)
        CASE
            WHEN traffic_source LIKE '%googlesyndication.com%' OR traffic_source LIKE '%doubleclick.net%'
                THEN 'googlesyndication.com (Display)'
            WHEN traffic_source IS NULL OR traffic_source = '(direct)'
                THEN '(direct)'
            ELSE traffic_source
        END AS cleaned_traffic_source,

        -- High-level Channel Grouping
        CASE
            WHEN traffic_source IS NULL OR traffic_source = '(direct)' THEN 'Direct'
            WHEN traffic_medium LIKE '%organic%' OR traffic_source IN ('google', 'bing', 'yahoo') THEN 'Organic Search'
            WHEN traffic_medium LIKE '%cpc%' OR traffic_medium LIKE '%ppc%' THEN 'Paid Search'
            WHEN traffic_source LIKE '%googlesyndication.com%' OR traffic_medium LIKE '%display%' THEN 'Display / Ads'
            ELSE 'Referral / Other'
        END AS channel_grouping

    FROM extracted
)

```
- Ideally the change would be implemented as business logic in intermediate, but I'm sticking with staging due to limited time.


### ML model

#### Training process
The following file is created to later include the query for creating the ML model.

```Bash
touch marketing_analytics/analyses/ml_model_training.sql
```
The model is trained as follows:
```SQL
CREATE OR REPLACE MODEL `crystalloids-analytics.ml.purcahse_model`
OPTIONS(
  model_type = 'logistic_reg',
  input_label_cols = ['label_made_purchase'],
  l1_reg = 0.1,
  l2_reg = 1.0
) AS

SELECT
    -- Features used for training
    total_sessions,
    total_events,
    COALESCE(device_category, 'unknown') AS device_category,
    COALESCE(country, 'unknown') AS country,
    
    -- Target Label (1 if they purchased, 0 otherwise)
    label_made_purchase

FROM `crystalloids-analytics.analytics.user_features`;
```
- Note that for a cleaner structure (and possibly different and independant access models), the ML model and its predictions table is materialized in a separate dataset `ml` while the `analytics` dataset is strictly kept for data analytics purposes and modeling.
- I chose for a loistic regression model since patterns related to predicting **whether or not** something happens (in this case, purchase) is best captured via this model type, which combined powerful binary representation and regression modeling.
- I have set regularization weights to avoid overfitting and/or high weights which only lead to numerical instability.
- I did not set maximum iteration (mostly BQ chooses 12) since BQ's internal algorithms often tune these parameteres best. If I see maximum iteration is reached, I would train with a higher threshhold.
- Ideally I woul perfom temporal training logic, but I stick with BQ's random sampling for now due to limited time.

#### Evaluation of the trained model
The following were observed after training the model:
- Itearions were 9, with maximum allowed 20. No need for new round.
- Precision 0.41 and Recall 0.18 shows we did not converge too a leaked or overfit model.
- ROC AUC is 0.98 which is too high for such low percision and recall values! Since other signs of leakage are not present, this may be coming from imbalanced source data. Therefore, training will be run with `auto_class_weights = TRUE`.
- With `auto_class_weights`, model converges after 7 iterations, precision 0.29 and recall 0.94 which is an improvement (since recall is often more important that precision in e-Commerce).
- For now, this is a satisfactory state, and I'lll move on to building the prediction table.
- The weights can be extracted via the following query:
```SQL
SELECT
  processed_input,
  weight,
  category_weights
FROM
  ML.WEIGHTS(MODEL `crystalloids-analytics.ml.purcahse_model`);
```

#### Prediction table

The prediction table is created in BigQuery via the following query:
```SQL
CREATE OR REPLACE TABLE `crystalloids-analytics.ml.purcahse_predictions` AS
SELECT
    user_pseudo_id,
    total_sessions,
    total_events,
    device_category,
    country,
    
    -- The binary prediction (0 or 1) made by the model
    predicted_label_made_purchase,
    
    -- Extract the exact probability (0.0 to 1.0) that the user will purchase
    (
      SELECT p.prob
      FROM UNNEST(predicted_label_made_purchase_probs) p
      WHERE p.label = 1
    ) AS purchase_probability

FROM
  ML.PREDICT(MODEL `crystalloids-analytics.ml.purcahse_model`, (
    SELECT
      user_pseudo_id,
      total_sessions,
      total_events,
      device_category,
      country,
      label_made_purchase
    FROM `crystalloids-analytics.analytics.user_features`
  ))
-- Order by highest purchase probability so marketing can target the top leads first
ORDER BY purchase_probability DESC;
```

- I then connected this table to a Looker Studio report for visualization for the future user. The report can be accessed [here](https://datastudio.google.com/reporting/aaded2ec-a3e7-44a2-865f-cb406c31fd77).
- The report modules enable cross-filtering for easier navigation.
- I have used a separate numbering signal and shadow background to visually separate reporting and filtering areas/handles.

## Key Decisions

### AI usage declaration
- Solely used Google Gemini due to time-bound test-case nature of the project.
- No text was copied into console/prompt box from the assignment doc due to security concerns.
- There was an error during installation of dbt due to incompatible dependancies, for which I copied the error codes in my prompts to help resolve them.

### Process reflections
- The following were new experience, which no always worked well regardless of getting effective help from AI and time-bound setting of expectations for myself:
  - setting up Colab/GitHub connection
  - using notebook as terminal was not a straightforward experience.
- I decided to skip creating intermediate models and orchestrating DAG to run dbt due to limited time.
- For the same reason (as above), plus limited experience with Terraform, i also decided to skip creating the project and database through IaC. Howvere, I understand this is a much more stabel (and elegant) approach.